# 00 - Preparación / compilación de bases EPH

**Este es el notebook base del proyecto.** Compila las bases de la EPH descargadas
manualmente del INDEC y subidas a **Google Drive** (carpeta `carga_EPH`): une la base de
**individuos** con la de **hogares** por el código de vínculo (`CODUSU` + `NRO_HOGAR`) y
deja los datasets listos para que **el resto de los notebooks (01-05) los procesen** sin
tener que volver a descargar ni unir las bases.

**Salida (en Google Drive, `carga_EPH/processed/`):**
- `eph_T<Q><YY>.parquet`: **un archivo por trimestre**, ya con individuos+hogares unidos
  y columnas `ANIO`/`TRIMESTRE`. Se guardan en Drive (persistentes entre sesiones). No se
  genera un único panel combinado: desbordaría la RAM de Colab. Los notebooks leen lo que
  necesitan con `load_panel(columns=..., quarters=..., out_dir=PROCESSED_DIR)`.

**Flujo:** Setup (clonar repo + montar Drive + carpeta de salida) → Diagnóstico de
`carga_EPH` → Trimestres disponibles → Compilar (1 parquet por trimestre, a Drive) →
Verificación (merge + montos numéricos + quiebre de esquema 4T2023).

**Cómo agregar un trimestre nuevo:**
1. Descargar del INDEC el `.zip` del trimestre (ej. `EPH_usu_4_Trim_2025_txt.zip`).
2. Subir el `.zip` (sin descomprimir) a Google Drive, carpeta `carga_EPH`.
3. Volver a correr este notebook (con `overwrite=False` compila solo lo que falte).

> ⚠️ **Quiebre de esquema en 4T2023.** Los trimestres ≤ T3-2023 tienen menos columnas
> (177 individual / 88 hogar) que los ≥ T4-2023 (235 / 98). Las variables nuevas
> (`EMPLEO`, `SECTOR`, ingresos/pensiones desagregados, deciles `P_DECCF`) no existen en
> los trimestres viejos. Ver `.claude/memoria_EPH.md`.

## 1. Setup (Colab)

Clona el repo (para `src/data_loader.py`), monta Google Drive y define la carpeta de
salida donde se guardan los parquets compilados.

In [ ]:
import sys, os

REPO_URL = "https://github.com/santiagoriverti/analisis_EPH.git"
REPO_DIR = "/content/analisis_EPH"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

# Carpeta de salida en Drive (persistente entre sesiones de Colab).
PROCESSED_DIR = "/content/drive/MyDrive/carga_EPH/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)
print("Parquets compilados se guardan en:", PROCESSED_DIR)

## 2. Diagnóstico: ¿qué hay en carga_EPH?

Muestra qué archivos detecta Colab en la carpeta de Drive y el contenido de un `.zip`
de ejemplo. Si la lista de trimestres (celda 3) sale vacía, revisar acá.

In [ ]:
from src.data_loader import DRIVE_DIR
import zipfile

print("DRIVE_DIR:", DRIVE_DIR, "| existe:", os.path.isdir(DRIVE_DIR))

if os.path.isdir(DRIVE_DIR):
    archivos = os.listdir(DRIVE_DIR)
    print(f"Archivos encontrados ({len(archivos)}):")
    for a in archivos[:10]:
        print(" -", a)

    zips = [a for a in archivos if a.lower().endswith(".zip")]
    if zips:
        ejemplo = os.path.join(DRIVE_DIR, zips[0])
        print(f"\nContenido de {zips[0]}:")
        with zipfile.ZipFile(ejemplo) as zf:
            for n in zf.namelist():
                print(" -", n)

## 3. Trimestres disponibles

Escanea `carga_EPH` y lista los trimestres detectados (con ambas bases, individual y hogar).

In [ ]:
from src.data_loader import list_available_quarters, _period_tag

available = list_available_quarters()
print("Disponibles:", [_period_tag(y, p) for y, p in available])

## 4. Compilación del panel (memory-safe, guarda en Drive)

Procesa **un trimestre por vez**: une individuos + hogares por `CODUSU` + `NRO_HOGAR`,
agrega `ANIO`/`TRIMESTRE` y guarda **un parquet por trimestre** en `PROCESSED_DIR` (Drive).

> No concatena los 36 trimestres en memoria (desbordaría la RAM de Colab). Los notebooks
> 01-05 leen lo que necesiten con `load_panel(columns=..., quarters=..., out_dir=PROCESSED_DIR)`.

In [ ]:
from src.data_loader import build_panel
import pandas as pd

# overwrite=True regenera todo. En próximas corridas usar overwrite=False para que
# saltee los trimestres que ya están compilados en Drive.
resumen = build_panel(available, out_dir=PROCESSED_DIR, overwrite=True)
df_resumen = pd.DataFrame(resumen)
print("Trimestres en resumen:", len(df_resumen))
print("Total de filas compiladas:", df_resumen["filas"].sum())
df_resumen[["year", "period", "filas", "columnas", "estado"]]

## 5. Verificación del merge y de los montos

Lee una muestra de un trimestre (solo columnas clave). `ITF`/`IPCF` deben ser **numéricos**
(IPCF debe verse como `2933333.33` con punto, no `2933333,33` con coma).

In [ ]:
from src.data_loader import load_panel

muestra = load_panel(
    columns=["CODUSU", "NRO_HOGAR", "COMPONENTE", "CH04", "CH06", "ESTADO", "ITF", "IPCF"],
    quarters=[(2025, 4)],
    out_dir=PROCESSED_DIR,
)
print("Shape muestra T4-2025:", muestra.shape)
print("dtype ITF:", muestra["ITF"].dtype, "| dtype IPCF:", muestra["IPCF"].dtype)
muestra.head()

## 6. Verificación del quiebre de esquema (4T2023)

Para cada variable del esquema nuevo, muestra el primer trimestre con datos no nulos
(debería ser 2023T4).

In [ ]:
vars_nuevas = ["EMPLEO", "SECTOR", "P_DECCF", "V2_01_M", "V5_01_M"]

chk = load_panel(columns=vars_nuevas, out_dir=PROCESSED_DIR)

for col in vars_nuevas:
    if col not in chk.columns:
        print(f"{col}: no está en ningún trimestre")
        continue
    con_datos = chk.loc[chk[col].notna(), ["ANIO", "TRIMESTRE"]]
    if con_datos.empty:
        print(f"{col}: sin datos en ningún trimestre")
    else:
        primer = con_datos.sort_values(["ANIO", "TRIMESTRE"]).iloc[0]
        print(f"{col}: primer trimestre con datos -> {int(primer.ANIO)}T{int(primer.TRIMESTRE)}")

## Cómo usan estos datos los notebooks 01-05

Cada notebook monta Drive, clona el repo y llama a `load_panel` apuntando a la carpeta de
Drive, pidiendo solo las columnas (y trimestres) que necesite:

```python
from src.data_loader import load_panel

PROCESSED_DIR = "/content/drive/MyDrive/carga_EPH/processed"

# Ejemplo (demografía): edad, sexo, región y ponderador, todos los trimestres
df = load_panel(columns=["CH06", "CH04", "REGION", "PONDERA"], out_dir=PROCESSED_DIR)

# Ejemplo (informalidad, solo esquema nuevo): restringir a T4-2023 en adelante
df = load_panel(
    columns=["EMPLEO", "SECTOR", "PONDERA"],
    quarters=[(2023, 4), (2024, 1), (2024, 2), (2024, 3), (2024, 4)],
    out_dir=PROCESSED_DIR,
)
```

Los parquets quedan en Drive (`carga_EPH/processed`): una vez corrido este notebook 00,
los demás leen directo de ahí sin recompilar.